# Principio de Segregación de Interfaces (ISP) — Cafetería

**Dominio propio:** equipos de la barra de la cafetería (molino, cafetera express, licuadora).

El ISP dice que un cliente no debe verse obligado a depender de métodos que no usa. Mejor varias interfaces pequeñas que una "gorda".

Muestro primero una interfaz gorda que obliga a implementar métodos sin sentido (viola) y luego interfaces segregadas (cumple).

## Versión que VIOLA el ISP

Una única interfaz `EquipoBarra` obliga a todo equipo a implementar `moler`, `extraer_espresso` y `licuar`, aunque cada máquina solo hace una parte.

In [1]:
from abc import ABC, abstractmethod

class EquipoBarra(ABC):
    @abstractmethod
    def moler(self, gramos: int) -> str: ...

    @abstractmethod
    def extraer_espresso(self, onzas: int) -> str: ...

    @abstractmethod
    def licuar(self, segundos: int) -> str: ...


class Molino(EquipoBarra):
    def __init__(self, marca: str) -> None:
        self.marca: str = marca
        self.usos: int = 0

    def moler(self, gramos: int) -> str:
        self.usos += 1
        return f"{self.marca} molio {gramos}g de cafe"

    def extraer_espresso(self, onzas: int) -> str:
        raise NotImplementedError("Un molino no extrae espresso")  # metodo forzado sin sentido

    def licuar(self, segundos: int) -> str:
        raise NotImplementedError("Un molino no licua")  # metodo forzado sin sentido

In [2]:
molino = Molino("Baratza")
print(molino.moler(18))
# El molino se ve OBLIGADO a tener estos metodos que no usa:
try:
    molino.licuar(30)
except NotImplementedError as e:
    print("Error esperado:", e)

Baratza molio 18g de cafe
Error esperado: Un molino no licua


### Problema

`Molino` depende de `extraer_espresso` y `licuar` que no tienen sentido para él. Se rellenan con `NotImplementedError`, lo que crea métodos trampa: el código cliente puede llamarlos y reventar. Viola el ISP.

## Versión que CUMPLE el ISP

Segrego la interfaz gorda en tres capacidades pequeñas: `Moledor`, `Espressor` y `Licuador`. Cada equipo implementa **solo** lo que realmente hace.

In [3]:
class Moledor(ABC):
    @abstractmethod
    def moler(self, gramos: int) -> str: ...

class Espressor(ABC):
    @abstractmethod
    def extraer_espresso(self, onzas: int) -> str: ...

class Licuador(ABC):
    @abstractmethod
    def licuar(self, segundos: int) -> str: ...


class Molino(Moledor):
    def __init__(self, marca: str) -> None:
        self.marca: str = marca
        self.usos: int = 0

    def moler(self, gramos: int) -> str:
        self.usos += 1
        return f"{self.marca} molio {gramos}g de cafe"

    def limpiar(self) -> str:
        return f"{self.marca} limpiado tras {self.usos} usos"


class MaquinaEspresso(Espressor):
    def __init__(self, marca: str) -> None:
        self.marca: str = marca
        self.presion_bar: int = 9

    def extraer_espresso(self, onzas: int) -> str:
        return f"{self.marca} extrajo {onzas}oz a {self.presion_bar} bar"

    def calentar(self) -> str:
        return f"{self.marca} precalentando el grupo"


class MolinoConEspresso(Moledor, Espressor):
    """Equipo combinado: implementa solo las capacidades que tiene."""
    def __init__(self, marca: str) -> None:
        self.marca: str = marca
        self.usos: int = 0

    def moler(self, gramos: int) -> str:
        self.usos += 1
        return f"{self.marca} molio {gramos}g"

    def extraer_espresso(self, onzas: int) -> str:
        return f"{self.marca} extrajo {onzas}oz"

In [4]:
# Cada cliente depende SOLO de la capacidad que necesita
def preparar_molienda(equipo: Moledor, gramos: int) -> None:
    print(equipo.moler(gramos))

def preparar_shot(equipo: Espressor, onzas: int) -> None:
    print(equipo.extraer_espresso(onzas))

molino = Molino("Baratza")
maquina = MaquinaEspresso("Rancilio")
combo = MolinoConEspresso("Breville")

preparar_molienda(molino, 18)
preparar_shot(maquina, 2)
preparar_molienda(combo, 20)
preparar_shot(combo, 1)

Baratza molio 18g de cafe
Rancilio extrajo 2oz a 9 bar
Breville molio 20g
Breville extrajo 1oz


### Análisis

Al segregar la interfaz, `Molino` ya no arrastra `extraer_espresso` ni `licuar`. Los clientes (`preparar_molienda`, `preparar_shot`) dependen solo de la capacidad que usan, y un equipo combinado como `MolinoConEspresso` compone únicamente las interfaces que le aplican. Nadie implementa métodos vacíos ni trampa.